# Preprocessing for topic modelling

### Topic modelling preprocessing pipeline overview

Input: `stylecom_cleaned.csv`
Output: `checkpoints/df_with_noun_tokens.pkl`

- **1. Setup:** load dependencies, define paths and column names, build a brand/designer lookup from the dataset metadata
- **2. Fashion ontology:** construct a vocabulary of fashion-relevant terms from two sources:
    - a curated seed list of nouns and adjectives
    - the [Fashionpedia](https://fashionpedia.github.io/) dataset 
    - lemmatise all terms and index them into unigram, bigram, and trigram lookup sets
- **3. POS tagging & token filtering:** normalise review text (expand contractions, strip punctuation), run spaCy POS tagging, then filter down to content-bearing nouns and adjectives
    - excluding stopwords, brand names, and domain-generic words such as "collection"
- **4. Checkpoint:** save the dataframe with per-document noun token list for further processing

### Key variables
- `ontology_df`: term → POS category lookup, used to guide token filtering
- `token_df`: one row per token, with lemma, POS, and brand flags
- `df["noun_tokens"]`: list of filtered lemmas per review, used as input to the topic model

In [7]:
import re
import string
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import json
import spacy
import matplotlib
matplotlib.use("Agg")

# 1. Preprocess data with SpaCy

In [ ]:
# Cell 1: configuration

DATA_PATH     = Path("data/stylecom_cleaned.csv")
ONTOLOGY_PATH = Path("data/fashion_ontology.csv")
OUT_DIR       = Path("eda_outputs")
CHECKPOINT_DIR = Path("checkpoints")
OUT_DIR.mkdir(exist_ok=True)

TEXT_COL      = "review"
BRAND_COL     = "designer"
DATE_COL      = "date"

In [8]:
# Cell 2: Load data

# Load from checkpoint (if this notebook was run before) or raw CSV
checkpoint_path = CHECKPOINT_DIR / "df_with_noun_tokens.pkl"

if checkpoint_path.exists():
    df = pd.read_pickle(checkpoint_path)
    print(f"Loaded from checkpoint: {df.shape}")
    print("Skip to notebook 3")
else:
    df = pd.read_csv(DATA_PATH).reset_index(names="doc_id")
    print(f"Loaded from CSV: {df.shape}")

Loaded from checkpoint: (6629, 13)
Skip to notebook 3


In [9]:
# Cell 3: Load raw data

print(df.shape)
print(df.columns.tolist())
df.head()

(6629, 13)
['doc_id', 'year', 'season', 'designer', 'author', 'city', 'date', 'review', '_date', 'review_norm', 'brand_mentions', 'noun_tokens', 'noun_text']


,doc_id,year,season,designer,author,city,date,review,_date,review_norm,brand_mentions,noun_tokens,noun_text
0,0,2000,Spring,Matt Nye,Armand Limnander,New York,17-Sep-99,Designer Matt Nye's sophomore show featured a ...,1999-09-17,Designer Matt Nye s sophomore show featured a ...,"[sophomore, matt nye]","[sophomore, coed, sailor, simple, cotton, polo...",sophomore coed sailor simple cotton polo shirt...
1,1,2000,Spring,Giorgio Armani,Armand Limnander,Milan,29-Sep-99,"Armani proposed a light, feminine silhouette f...",1999-09-29,Armani proposed a light feminine silhouette fo...,[sea],"[light, feminine, millennium, foam, fuchsia, l...",light feminine millennium foam fuchsia lime ch...
2,2,2000,Spring,Eric Bergère,Armand Limnander,Paris,4-Oct-99,Broadway Garnier was the theme for Eric Berg'r...,1999-10-04,Broadway Garnier was the theme for Eric Berg a...,[],"[theme, tailleur, pleat, sweater, ruched, shir...",theme tailleur pleat sweater ruched shirt inte...
3,3,2000,Spring,Céline,Armand Limnander,Paris,7-Oct-99,Getaway glamour was the theme for Celine's str...,1999-10-07,Getaway glamour was the theme for Celine s str...,"[michael kors, trademark]","[glamour, theme, presentation, course, fun, su...",glamour theme presentation course fun sun spea...
4,4,2000,Spring,Byblos,Armand Limnander,Milan,27-Sep-99,Judo Jetson blends my favorite cartoon charact...,1999-09-27,Judo Jetson blends my favorite cartoon charact...,"[john bartlett, byblos]","[favorite, cartoon, character, spiritual, japa...",favorite cartoon character spiritual japanese ...


In [10]:
# Cell 4: Build brand lookup from metadata

brand_names_from_metadata = (
    df[BRAND_COL]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

CUSTOM_BRANDS = [
    # Will add any additional brand/designer names if needed.
]

BRAND_NAMES  = sorted(set(brand_names_from_metadata) | set(CUSTOM_BRANDS))
BRAND_PHRASES = {b.lower() for b in BRAND_NAMES}
BRAND_WORDS   = {
    token.lower()
    for brand in BRAND_NAMES
    for token in re.findall(r"\b\w+\b", str(brand))
}

print(f"{len(BRAND_NAMES)} brand/designer phrases")
BRAND_NAMES[:10]

815 brand/designer phrases


['1 Piu 1 Uguale 3',
 '10 Crosby Derek Lam',
 '3.1 Phillip Lim',
 '6267',
 'A Degree Fahrenheit',
 'A Détacher',
 'A.F. Vandevorst',
 'A.L.C.',
 'A.P.C.',
 'A.W.A.K.E.']

# 2. Building a Fashion Ontology + SpaCy setup

In [ ]:
# Cell 5: Download fashionpedia data 

url = "https://s3.amazonaws.com/ifashionist-dataset/annotations/instances_attributes_val2020.json"
response = requests.get(url)

with open("data/instances_attributes_val2020.json", "wb") as f:
    f.write(response.content)

In [ ]:
# Cell 6: Fashionpedia supercategories to noun/adjective
SUPERCATEGORY_POS_MAP = {
    # Categories: all nouns
    "upperbody": "noun",
    "lowerbody": "noun",
    "wholebody": "noun",
    "accessories": "noun",
    "parts": "noun",

    # Attributes: adjectives (style/shape descriptors)
    "nickname": "adjective",
    "silhouette": "adjective",
    "neckline": "adjective",
    "collar": "adjective",
    "sleeve": "adjective",
    "closure": "adjective",
    "fit": "adjective",
    "pattern": "adjective",
    "color": "noun",
    "material": "noun",
    "technique": "adjective",
}

In [ ]:
# Cell 7: Load fashionpedia data, build ontology dataframe
with open("data/instances_attributes_val2020.json") as f:
    data = json.load(f)

rows = []

for cat in data["categories"]:
    terms = [t.strip() for t in cat["name"].split(",")]
    for term in terms:
        rows.append({
            "term": term.lower().strip(),
            "category": SUPERCATEGORY_POS_MAP.get(cat["supercategory"].lower(), "noun")
        })

for attr in data["attributes"]:
    term = re.sub(r"\s*\(.*?\)", "", attr["name"]).strip()
    rows.append({
        "term": term.lower().strip(),
        "category": SUPERCATEGORY_POS_MAP.get(attr["supercategory"].lower(), "adjective")
    })

fashionpedia_df = (
    pd.DataFrame(rows)
    .drop_duplicates(subset=["term"])
    .query("term != ''")
)

print(fashionpedia_df["category"].value_counts())
fashionpedia_df.head(5)

category
adjective    254
noun          53
Name: count, dtype: int64


,term,category
0,shirt,noun
1,blouse,noun
2,top,noun
3,t-shirt,noun
4,sweatshirt,noun


In [ ]:
# Cell 8: Add fashionpedia ontology to data

fashionpedia_df.to_csv("data/fashionpedia_ontology.csv", index=False)

In [ ]:
# Cell 9: Handmade fashion vocabulary
# I've used this to add any additional fashion-relevant terms that are not in the Fashionpedia ontology

FASHION_ADJECTIVES = {
    "tailored", "structured", "sheer", "pleated", "draped", "cropped", "oversized",
    "fitted", "fluid", "minimal", "romantic", "sporty", "luxurious", "ornate",
    "embellished", "textured", "sleek", "feminine", "masculine", "voluminous",
    "layered", "clean", "sharp", "soft", "opulent", "subtle", "graphic",
    "monochrome", "vintage", "modern", "sculptural", "elegant", "casual",
    "deconstructed", "asymmetrical", "slouchy", "lean", "dramatic", "polished"
}

FASHION_NOUNS = {
    "dress", "skirt", "coat", "jacket", "trouser", "pants", "shirt", "blouse",
    "tunic", "gown", "suit", "sweater", "knit", "cardigan", "top", "vest",
    "cape", "parka", "anorak", "bodice", "hem", "silhouette", "waist", "lapel",
    "sleeve", "collar", "cuff", "pocket", "seam", "panel", "fabric", "textile",
    "silk", "wool", "cotton", "satin", "jersey", "lace", "tweed", "leather",
    "denim", "velvet", "chiffon", "organza", "brocade", "cashmere", "print",
    "pattern", "embroidery", "bead", "sequin", "fringe", "pleat", "ruffle",
    "collection", "look", "runway", "couture", "accessory", "boot", "shoe",
    "heel", "sandal", "bag", "belt", "glove", "hat", "scarf", "garment",
    "lining", "button", "zip", "zipper", "trim", "tulle", "crepe"
}


FASHION_TERMS = None

# Set to True to count brand/designer names as fashion-relevant terms
COUNT_BRANDS_AS_FASHION = True

### SpaCy setup

In [ ]:
# Cell 10: spaCy setup

try:
    nlp = spacy.load("en_core_web_sm", disable=["ner"])
except OSError:
    raise OSError(
        "spaCy model not installed. Run:\n  python -m spacy download en_core_web_sm"
    )

nlp.max_length = max(2_000_000, nlp.max_length)

In [ ]:
# Cell 11: Build ontology dataframe

ONTOLOGY_PATH = None

if ONTOLOGY_PATH is not None:
    ontology_df = pd.read_csv(ONTOLOGY_PATH)
    ontology_df["term"]     = ontology_df["term"].astype(str).str.strip().str.lower()
    ontology_df["category"] = ontology_df["category"].astype(str).str.strip().str.lower()
else:
    seed_rows = (
        [(t.lower(), "adjective") for t in FASHION_ADJECTIVES] +
        [(t.lower(), "noun")      for t in FASHION_NOUNS]
    )
    if COUNT_BRANDS_AS_FASHION:
        seed_rows += [(t.lower(), "brand") for t in BRAND_PHRASES]
    ontology_df = pd.DataFrame(seed_rows, columns=["term", "category"])

if FASHION_TERMS:
    custom_df   = pd.DataFrame(
        [(k.lower().strip(), v.lower().strip()) for k, v in CUSTOM_FASHION_TERMS.items()],
        columns=["term", "category"]
    )
    ontology_df = pd.concat([ontology_df, custom_df], ignore_index=True)

ontology_df = ontology_df.drop_duplicates(subset=["term"]).copy()
ontology_df["n_words"] = ontology_df["term"].str.split().str.len()


In [ ]:
# Cell 12: Lemmatise ontology terms, drop duplicates again
def lemmatise_term(term: str) -> str:
    doc = nlp(term)
    return " ".join(t.lemma_.lower() for t in doc)

ontology_df["term"] = ontology_df["term"].apply(lemmatise_term)
ontology_df = ontology_df.drop_duplicates(subset=["term"]).copy()

In [ ]:
# Cell 13: Derive lookup sets from ontology dataframe
ONTOLOGY_UNIGRAMS = set(ontology_df.loc[ontology_df["n_words"] == 1, "term"])
ONTOLOGY_BIGRAMS  = set(ontology_df.loc[ontology_df["n_words"] == 2, "term"])
ONTOLOGY_TRIGRAMS = set(ontology_df.loc[ontology_df["n_words"] == 3, "term"])
TERM_TO_CATEGORY  = dict(zip(ontology_df["term"], ontology_df["category"]))

print(f"Ontology: {len(ONTOLOGY_UNIGRAMS)} unigrams, {len(ONTOLOGY_BIGRAMS)} bigrams, {len(ONTOLOGY_TRIGRAMS)} trigrams")
ontology_df.sample(10)

Ontology: 357 unigrams, 446 bigrams, 87 trigrams


,term,category,n_words
449,miu miu,brand,2
736,erdem,brand,1
35,sheer,adjective,1
176,mint design,brand,2
882,commuun,brand,1
833,we be handsome,brand,3
283,jason wu,brand,2
892,riccardo tisci,brand,2
642,cerre,brand,1
775,veronica b. vallenes,brand,3


In [ ]:
# Cell 14: Save ontology dataframe to CSV

ontology_df.to_csv("data/fashion_ontology.csv", index = False)

# 3. POS tagging

In [ ]:
# Cell 15: Load data
df = pd.read_csv(DATA_PATH).reset_index(names="doc_id")


df["_date"] = pd.to_datetime(df[DATE_COL], format="%d-%b-%y")

print(df.shape)
print(f"Date range: {df['_date'].min().date()} → {df['_date'].max().date()}")

(6629, 9)
Date range: 1999-09-12 → 2014-06-12


In [ ]:
# Cell 16: Text normalisation

CONTRACTION_MAP = {
    r"\bcan['\'']t\b": "cannot",
    r"\bwon['\'']t\b": "will not",
    r"n['\'']t\b":       " not",
    r"['\'']re\b":       " are",
    r"['\'']ve\b":       " have",
    r"['\'']ll\b":       " will",
    r"['\'']d\b":        " would",
    r"['\'']m\b":        " am",
    r"['\'']s\b":        " s",
}

punctuation_to_space = str.maketrans({p: " " for p in string.punctuation})

def normalize_text(text: str) -> str:
    text = str(text)
    for pattern, repl in CONTRACTION_MAP.items():
        text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
    text = text.translate(punctuation_to_space)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["review_norm"] = df[TEXT_COL].fillna("").apply(normalize_text)
df[[TEXT_COL, "review_norm"]].head(2)

In [ ]:
# Cell 17: Brand mention extraction (preserves brand info separately from model text)

def extract_brand_mentions(text: str, brand_phrases=BRAND_PHRASES):
    text_l = str(text).lower()
    return sorted(
        [b for b in brand_phrases if re.search(rf"\b{re.escape(b)}\b", text_l)],
        key=len, reverse=True
    )

df["brand_mentions"] = df[TEXT_COL].fillna("").apply(extract_brand_mentions)
df[[BRAND_COL, "brand_mentions"]].head(10)

In [ ]:
# Cell 18: POS tagging on normalised text

def spacy_process(texts, batch_size=64):
    rows = []
    for doc_id, doc in zip(df["doc_id"], nlp.pipe(texts, batch_size=batch_size)):
        for token in doc:
            if token.is_space:
                continue
            rows.append({
                "doc_id":        doc_id,
                "token":         token.text,
                "lemma":         token.lemma_.lower().strip(),
                "pos":           token.pos_,
                "tag":           token.tag_,
                "is_alpha":      token.is_alpha,
                "is_stop":       token.is_stop,
                "is_brand_word": token.text.lower() in BRAND_WORDS,
            })
    return pd.DataFrame(rows)
token_df = spacy_process(df["review_norm"].tolist())
token_df.head()

In [ ]:
# Cell 19: Filter tokens: keep non-stop nouns, drop domain-generic nouns

VALID_POS_FOR_MODELLING = {"NOUN", "ADJ"}

# Runway-review generic nouns that carry no topical signal
DOMAIN_STOP_NOUNS = {
    "collection", "collections", "season", "show", "shows", "look", "looks",
    "designer", "designers", "way", "ways", "thing", "things", "idea", "ideas",
    "version", "versions", "kind", "kinds", "sort", "sorts", "piece", "pieces",
    "mix", "moment", "moments", "sense", "touch", "reference", "references",
    "view", "views", "line", "lines", "woman", "women", "girl", "girls",
    "man", "men", "time", "year", "years", "day", "days", "world", "dress", 
    "jacket", "skirt", "coat", "black", "print", "fashion",
    "clothe", "new", "color", "fabric", "leather", "white", 
    "runway", "pant", "silk", "good", "short", "model", "today", "little", "silhouette", "gown",
    "style"
}

token_df = token_df[
    token_df["is_alpha"] &
    token_df["lemma"].ne("") &
    token_df["pos"].isin(VALID_POS_FOR_MODELLING) &
    (~token_df["is_stop"]) &
    (token_df["lemma"].str.len() >= 3) &
    (~token_df["lemma"].isin(DOMAIN_STOP_NOUNS))
].copy()

token_df.head()

In [ ]:
# Cell 20: Build per-document noun token lists

doc_nouns = (
    token_df.groupby("doc_id")["lemma"]
    .apply(list)
    .rename("noun_tokens")
    .reset_index()
)

df = df.merge(doc_nouns, on="doc_id", how="left")
df["noun_tokens"] = df["noun_tokens"].apply(lambda x: x if isinstance(x, list) else [])
df["noun_text"]   = df["noun_tokens"].apply(lambda toks: " ".join(toks))

df[["doc_id", "noun_tokens", "noun_text"]].head(3)

,doc_id,noun_tokens,noun_text
0,0,"[sophomore, coed, sailor, simple, cotton, polo...",sophomore coed sailor simple cotton polo shirt...
1,1,"[light, feminine, millennium, foam, fuchsia, l...",light feminine millennium foam fuchsia lime ch...
2,2,"[theme, tailleur, pleat, sweater, ruched, shir...",theme tailleur pleat sweater ruched shirt inte...


# 4. Checkpoint: save output

In [ ]:
# Cell 21: Save preprocessed df with noun tokens

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

df.to_pickle(CHECKPOINT_DIR / "df_with_noun_tokens.pkl")
print(f"Checkpoint saved: {CHECKPOINT_DIR / 'df_with_noun_tokens.pkl'}")

Checkpoint saved: checkpoints/df_with_noun_tokens.pkl
